# Global CLIP Evaluation on MVTec AD
Automated zero-shot evaluation across all 15 categories.

In [ ]:
!pip install "anomalib[vlm,clip]" pandas open_clip_torch

In [ ]:
import pandas as pd
from anomalib.models import WinClip
from anomalib.engine import Engine
from anomalib.data import MVTecAD
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

DATA_ROOT = "/data/mvtec-anomaly-detection"
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 8

all_results = []
engine = Engine()

for category in CATEGORIES:
    print(f"\n{'='*50}")
    print(f"Evaluating category: {category.upper()}")
    print(f"{'='*50}")
    
    datamodule = MVTecAD(
        root=DATA_ROOT,
        category=category,
        eval_batch_size=EVAL_BATCH_SIZE,
        num_workers=NUM_WORKERS
    )
    
    model = WinClip(class_name=category)
    results = engine.test(model=model, datamodule=datamodule)
    
    if results:
        res = results[0] if isinstance(results, list) else results
        res['category'] = category
        all_results.append(res)

if all_results:
    df = pd.DataFrame(all_results)
    display(df)

In [ ]:
import os
import json

SAVE_DIR = "/content/drive/MyDrive/ralm" if os.path.exists("/content/drive/MyDrive/ralm") else "../results"
os.makedirs(os.path.join(SAVE_DIR, "metrics"), exist_ok=True)

if all_results:
    per_category = {}
    macro_metrics = {"image_AUROC": 0, "image_F1Score": 0, "pixel_AUROC": 0, "pixel_F1Score": 0}
    
    for res in all_results:
        cat = res["category"]
        cat_dict = {
            "image_AUROC": res.get("image_AUROC", 0) * 100,
            "image_F1Score": res.get("image_F1Score", 0) * 100,
            "pixel_AUROC": res.get("pixel_AUROC", 0) * 100,
            "pixel_F1Score": res.get("pixel_F1Score", 0) * 100
        }
        per_category[cat] = cat_dict
        for k in macro_metrics:
            macro_metrics[k] += cat_dict[k]
            
    for k in macro_metrics:
        macro_metrics[k] /= len(all_results)
        
    result_json = {
        "method": "Anomalib_WinCLIP",
        "macro_metrics": {k: round(v, 2) for k, v in macro_metrics.items()},
        "per_category": {k: {m_k: round(m_v, 2) for m_k, m_v in v.items()} for k, v in per_category.items()}
    }
    
    json_path = os.path.join(SAVE_DIR, "metrics", "result_winclip.json")
    with open(json_path, "w") as f:
        json.dump(result_json, f, indent=2)
        
    csv_path = os.path.join(SAVE_DIR, "metrics", "result_winclip.csv")
    rows = [{"Category": k, **v} for k, v in per_category.items()]
    df_save = pd.DataFrame(rows)
    df_save.loc[len(df_save)] = ["MEAN", *[macro_metrics[m] for m in df_save.columns if m != "Category"]]
    df_save.to_csv(csv_path, index=False)
    print(f"WinCLIP results strictly saved to {SAVE_DIR}/metrics/")
